# Repository guide: OmniASR full training

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# OmniASR-W2V-300M → Tarifit V1.2 Fine-Tuning

This notebook performs the next controlled model-family experiment for the Tarifit ASR TFM.

**Model:** `ylacombe/omniASR_W2V_300M_SSL`  
**Task:** CTC fine-tuning on the frozen Tarifit V1.2 corpus  
**Train:** 1,754 segments (~5.22 h)  
**Validation:** 129 speaker-held-out segments  
**Augmentation:** none  
**Decoding:** greedy CTC  
**Selection metric:** validation CER

The purpose is to compare a second pretrained Wav2Vec2-style backbone with the MMS V1.2 systems while using the same final orthography, tokenizer inventory, train/validation split, and evaluation metrics.

Unlike MMS adapter fine-tuning, this experiment fine-tunes the OmniASR encoder while freezing only the convolutional feature extractor. Therefore it uses a lower learning rate and allows up to 8 epochs with early stopping.


In [ ]:
# Cell 1 — Install the exact experiment dependencies

!pip -q install \
    "transformers==4.57.1" \
    "datasets==4.4.1" \
    "accelerate>=1.10,<2" \
    "jiwer==4.0.0" \
    "safetensors>=0.4.5" \
    "soundfile>=0.12.1"

print("✓ Dependencies installed.")
print("If Colab requests a restart after installation, restart once and continue from Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✓ Dependencies installed.
If Colab requests a restart after installation, restart once and continue from Cell 2.


In [ ]:
# Cell 2 — Mount Drive and define experiment paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2_train_val_frozen.csv"
)

TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"

# Reuse the exact V1.2 cached waveforms + CTC labels.
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

OUTPUT_DIR = PROJECT_ROOT / "models" / "omniASR_w2v_300m_tarifit_v1_2"
RESULTS_DIR = PROJECT_ROOT / "results" / "omniASR_w2v_300m_tarifit_v1_2"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = "ylacombe/omniASR_W2V_300M_SSL"

print("Project:", PROJECT_ROOT)
print("Model:", BASE_MODEL_ID)
print("Output:", OUTPUT_DIR)
print("Results:", RESULTS_DIR)


Mounted at /content/drive
Project: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
Model: ylacombe/omniASR_W2V_300M_SSL
Output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2
Results: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/omniASR_w2v_300m_tarifit_v1_2


In [ ]:
# Cell 3 — Verify software versions and GPU

import sys
import json
import random
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import transformers
import datasets

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

assert transformers.__version__ == "4.57.1"
assert datasets.__version__ == "4.4.1"

if not torch.cuda.is_available():
    raise RuntimeError("Please switch Colab to a GPU runtime before continuing.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# Cell 4 — Load the exact frozen V1.2 train/validation split

assert FROZEN_METADATA_PATH.exists(), (
    f"Frozen metadata not found: {FROZEN_METADATA_PATH}"
)

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

for col in [
    "transcription",
    "final_selection",
    "review_status",
    "dataset_split",
]:
    if col in frozen_df.columns:
        frozen_df[col] = (
            frozen_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

frozen_df["dataset_split"] = frozen_df["dataset_split"].str.lower()

train_selected = frozen_df[
    frozen_df["dataset_split"].eq("train")
].copy()

val_selected = frozen_df[
    frozen_df["dataset_split"].eq("validation")
].copy()

selected = pd.concat(
    [train_selected, val_selected],
    ignore_index=True,
)

assert len(train_selected) == 1754, (
    f"Expected 1754 train segments, found {len(train_selected)}"
)
assert len(val_selected) == 129, (
    f"Expected 129 validation segments, found {len(val_selected)}"
)
assert len(selected) == 1883
assert not selected["segment_id"].duplicated().any()

print("Train segments:", len(train_selected))
print("Validation segments:", len(val_selected))
print(
    "Train hours:",
    round(train_selected["duration_seconds"].sum() / 3600, 3),
)
print(
    "Validation hours:",
    round(val_selected["duration_seconds"].sum() / 3600, 3),
)
print("Train speakers:", train_selected["speaker_group_id"].nunique())
print("Validation speakers:", val_selected["speaker_group_id"].nunique())

print("\n✓ Exact frozen V1.2 split loaded.")


Train segments: 1754
Validation segments: 129
Train hours: 5.223
Validation hours: 0.298
Train speakers: 3
Validation speakers: 2

✓ Exact frozen V1.2 split loaded.


In [ ]:
# Cell 5 — Verify speaker independence and test isolation

train_speakers = set(train_selected["speaker_group_id"].dropna())
val_speakers = set(val_selected["speaker_group_id"].dropna())

print("Train/validation speaker overlap:", bool(train_speakers & val_speakers))

assert not (train_speakers & val_speakers), (
    f"Speaker leakage detected: {train_speakers & val_speakers}"
)

# Inspect the master metadata only for held-out test speakers.
master_df = pd.read_csv(METADATA_PATH)

for col in ["dataset_split", "speaker_group_id"]:
    master_df[col] = (
        master_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

test_speakers = set(
    master_df.loc[
        master_df["dataset_split"].str.lower().eq("test"),
        "speaker_group_id",
    ]
)

print("Train/test speaker overlap:", bool(train_speakers & test_speakers))
print("Validation/test speaker overlap:", bool(val_speakers & test_speakers))

assert not (train_speakers & test_speakers)
assert not (val_speakers & test_speakers)

print("✓ No speaker leakage.")


Train/validation speaker overlap: False
Train/test speaker overlap: False
Validation/test speaker overlap: False
✓ No speaker leakage.


In [ ]:
# Cell 6 — Verify the final V1.2 character inventory

import unicodedata
from collections import Counter

FINAL_LETTERS = [
    "a", "b", "c", "d", "ḍ", "e", "ɛ", "f", "g", "h", "ḥ",
    "i", "j", "k", "l", "m", "n", "p", "q", "r", "s", "t",
    "ṭ", "u", "v", "w", "x", "y", "z", "ɣ", "ʷ",
]

allowed_chars = set(FINAL_LETTERS) | {" "}

counter = Counter(
    ch
    for text in selected["transcription"].astype(str)
    for ch in text
)

unexpected = sorted(
    ch
    for ch in counter
    if ch not in allowed_chars
)

combining_marks = {
    ch: count
    for ch, count in counter.items()
    if unicodedata.combining(ch)
}

print("Final letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected)
print("Combining marks:", combining_marks)

assert not unexpected
assert not combining_marks

print("✓ Final V1.2 orthography verified.")


Final letters: 31
Unexpected characters: []
Combining marks: {}
✓ Final V1.2 orthography verified.


In [ ]:
# Cell 7 — Verify frozen metadata hash

sha256 = hashlib.sha256(
    FROZEN_METADATA_PATH.read_bytes()
).hexdigest()

print("Frozen rows:", len(frozen_df))
print("SHA256:", sha256)
print("✓ Frozen metadata fingerprint recorded.")


Frozen rows: 1883
SHA256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab
✓ Frozen metadata fingerprint recorded.


In [ ]:
# Cell 8 — Load and verify the exact existing V1.2 tokenizer

import json

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

assert VOCAB_PATH.exists(), f"Missing tokenizer: {VOCAB_PATH}"

# ---------------------------------------------------------
# Load the exact vocabulary used to build the V1.2 dataset
# ---------------------------------------------------------

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    existing_vocab = json.load(f)

print("Saved vocabulary:")
print(existing_vocab)

print("\nVocabulary size:", len(existing_vocab))


# ---------------------------------------------------------
# Verify CONTENT, not arbitrary token-ID ordering
# ---------------------------------------------------------

expected_tokens = set(FINAL_LETTERS) | {
    "|",
    "[UNK]",
    "[PAD]",
}

saved_tokens = set(existing_vocab.keys())

missing_tokens = sorted(expected_tokens - saved_tokens)
unexpected_tokens = sorted(saved_tokens - expected_tokens)

print("\nMissing tokens:", missing_tokens)
print("Unexpected tokens:", unexpected_tokens)

assert len(existing_vocab) == 34, (
    f"Expected 34 tokens, found {len(existing_vocab)}"
)

assert not missing_tokens, (
    f"Missing expected tokens: {missing_tokens}"
)

assert not unexpected_tokens, (
    f"Unexpected tokenizer tokens: {unexpected_tokens}"
)

# IDs should form one unique 0..33 vocabulary
saved_ids = sorted(existing_vocab.values())

assert saved_ids == list(range(34)), (
    f"Unexpected token IDs: {saved_ids}"
)


# ---------------------------------------------------------
# Load tokenizer using the EXISTING IDs
# ---------------------------------------------------------

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


print("\nTokenizer size:", len(tokenizer))
print("PAD / CTC blank id:", tokenizer.pad_token_id)
print("UNK id:", tokenizer.unk_token_id)
print("Word delimiter | id:", tokenizer.convert_tokens_to_ids("|"))

assert len(tokenizer) == 34

print("\n✓ Exact existing V1.2 tokenizer reused successfully.")

Saved vocabulary:
{'[PAD]': 33, '[UNK]': 32, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '|': 0, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṭ': 31}

Vocabulary size: 34

Missing tokens: []
Unexpected tokens: []

Tokenizer size: 34
PAD / CTC blank id: 33
UNK id: 32
Word delimiter | id: 0

✓ Exact existing V1.2 tokenizer reused successfully.


In [ ]:
# Cell 9 — Verify zero unknown tokens in the frozen references

unk_id = tokenizer.unk_token_id
unknown_segments = []

for row in frozen_df.itertuples(index=False):
    ids = tokenizer(row.transcription).input_ids
    if unk_id in ids:
        unknown_segments.append(
            (row.segment_id, row.transcription)
        )

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    display(pd.DataFrame(
        unknown_segments[:30],
        columns=["segment_id", "transcription"],
    ))

assert not unknown_segments

print("✓ All references are representable by the tokenizer.")


Segments containing [UNK]: 0
✓ All references are representable by the tokenizer.


In [ ]:
# Cell 10 — Reuse the exact V1.2 cached waveforms and labels

from datasets import load_from_disk

assert DATASET_CACHE_DIR.exists(), (
    f"Missing cached dataset: {DATASET_CACHE_DIR}"
)
assert CACHE_MANIFEST_PATH.exists(), (
    f"Missing cache manifest: {CACHE_MANIFEST_PATH}"
)

with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as f:
    cache_manifest = json.load(f)

assert cache_manifest.get("metadata_sha256") == sha256, (
    "Cached dataset does not match the frozen metadata."
)

dataset = load_from_disk(str(DATASET_CACHE_DIR))
train_ds = dataset["train"]
val_ds = dataset["validation"]

assert len(train_ds) == 1754
assert len(val_ds) == 129

print("Train examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("✓ Exact V1.2 cached data reused.")


Train examples: 1754
Validation examples: 129
Columns: ['segment_id', 'input_values', 'input_length', 'labels']
✓ Exact V1.2 cached data reused.


In [ ]:
# Cell 11 — Show duration statistics

def duration_summary(frame):
    seconds = frame["duration_seconds"].astype(float)

    return {
        "segments": len(frame),
        "hours": seconds.sum() / 3600,
        "min_seconds": seconds.min(),
        "mean_seconds": seconds.mean(),
        "max_seconds": seconds.max(),
    }

train_duration = duration_summary(train_selected)
val_duration = duration_summary(val_selected)

display(pd.DataFrame([
    {"split": "train", **train_duration},
    {"split": "validation", **val_duration},
]))


,split,segments,hours,min_seconds,mean_seconds,max_seconds
0,train,1754,5.222733,0.848,10.719407,19.984
1,validation,129,0.298358,0.944,8.326264,26.000


In [ ]:
# Cell 12 — Load OmniASR-W2V-300M with a fresh Tarifit CTC head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,

    # Fresh Tarifit CTC output vocabulary
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,

    # No augmentation in this experiment
    apply_spec_augment=False,

    # Conservative regularization for full encoder fine-tuning
    attention_dropout=0.05,
    hidden_dropout=0.05,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
)

# Freeze only the convolutional acoustic feature extractor.
# The Transformer encoder + fresh CTC head remain trainable.
model.freeze_feature_encoder()

model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%",
)
print("CTC vocabulary size:", model.config.vocab_size)
print("SpecAugment enabled:", model.config.apply_spec_augment)

assert model.config.vocab_size == 34
assert model.config.apply_spec_augment is False

print("\n✓ OmniASR initialized with fresh Tarifit CTC head.")
print("✓ Convolutional feature extractor frozen; encoder + head trainable.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at ylacombe/omniASR_W2V_300M_SSL and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 315,473,570
Trainable parameters: 311,263,394
Trainable percentage: 98.6654%
CTC vocabulary size: 34
SpecAugment enabled: False

✓ OmniASR initialized with fresh Tarifit CTC head.
✓ Convolutional feature extractor frozen; encoder + head trainable.


In [ ]:
# Cell 13 — Run exact CTC feasibility checks with the OmniASR frame rate

def minimum_ctc_frames(labels):
    repeats = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats


def model_output_frames(input_samples):
    return int(
        model._get_feat_extract_output_lengths(
            torch.tensor(int(input_samples))
        ).item()
    )


def find_ctc_infeasible(split_ds):
    bad = []

    for i, example in enumerate(split_ds):
        output_frames = model_output_frames(
            example["input_length"]
        )

        min_frames = minimum_ctc_frames(
            example["labels"]
        )

        if output_frames < min_frames:
            bad.append({
                "index": i,
                "segment_id": example["segment_id"],
                "input_samples": example["input_length"],
                "audio_seconds": example["input_length"] / 16000,
                "output_frames": output_frames,
                "label_length": len(example["labels"]),
                "minimum_ctc_frames": min_frames,
            })

    return bad


bad_train = find_ctc_infeasible(train_ds)
bad_val = find_ctc_infeasible(val_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_val))

if bad_train:
    display(pd.DataFrame(bad_train))
if bad_val:
    display(pd.DataFrame(bad_val))

assert not bad_train, (
    "Training contains CTC-infeasible examples."
)
assert not bad_val, (
    "Validation contains CTC-infeasible examples."
)

print("✓ All 1754/129 examples are CTC-feasible for OmniASR.")


CTC-infeasible training examples: 0
CTC-infeasible validation examples: 0
✓ All 1754/129 examples are CTC-feasible for OmniASR.


In [ ]:
# Cell 14 — Define the dynamic CTC padding collator

from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        batch["labels"] = labels_batch[
            "input_ids"
        ].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

print("✓ CTC collator ready.")


✓ CTC collator ready.


In [ ]:
# Cell 15 — Define WER and CER

from jiwer import wer, cer

def compute_metrics(pred):
    pred_ids = np.argmax(
        pred.predictions,
        axis=-1,
    )

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = [
        x.strip()
        for x in processor.batch_decode(pred_ids)
    ]

    ref_str = [
        x.strip()
        for x in processor.batch_decode(
            label_ids,
            group_tokens=False,
        )
    ]

    return {
        "wer": wer(ref_str, pred_str),
        "cer": cer(ref_str, pred_str),
    }

print("✓ WER/CER metric function ready.")


✓ WER/CER metric function ready.


In [ ]:
# Cell 16 — Set reproducibility seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


Seed: 42


In [ ]:
# Cell 17 — Configure resumable OmniASR fine-tuning

from transformers import (
    TrainingArguments,
    EarlyStoppingCallback,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Full encoder adaptation can converge more slowly than MMS adapters.
    num_train_epochs=8,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    # Keep enough state to resume after a Colab interruption.
    save_total_limit=2,

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    dataloader_num_workers=2,
    remove_unused_columns=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

print("Max epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Weight decay:", training_args.weight_decay)
print("Effective batch size:", 2 * 8)
print("Selection metric:", training_args.metric_for_best_model)
print("Early-stopping patience:", 2)

print("✓ Training configuration ready.")


Max epochs: 8
Learning rate: 3e-05
Weight decay: 0.01
Effective batch size: 16
Selection metric: cer
Early-stopping patience: 2
✓ Training configuration ready.


In [ ]:
# Cell 18 — Create the OmniASR Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[early_stopping],
)

print("Trainer train examples:", len(trainer.train_dataset))
print("Trainer validation examples:", len(trainer.eval_dataset))

assert len(trainer.train_dataset) == 1754
assert len(trainer.eval_dataset) == 129

print("✓ OmniASR Trainer ready.")


Trainer train examples: 1754
Trainer validation examples: 129
✓ OmniASR Trainer ready.


In [ ]:
# Cell 19 — Start or resume OmniASR fine-tuning

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None

if OUTPUT_DIR.exists():
    last_checkpoint = get_last_checkpoint(
        str(OUTPUT_DIR)
    )

if last_checkpoint is not None:
    print("Found checkpoint:", last_checkpoint)
    print("Resuming training from the latest checkpoint.")

    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("No previous checkpoint found.")
    print("Starting fresh OmniASR V1.2 fine-tuning.")

    train_result = trainer.train()

print("\nTraining finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


No previous checkpoint found.
Starting fresh OmniASR V1.2 fine-tuning.


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,2.973300,3.264127,0.997449,0.991961
2,1.298900,3.276336,1.000425,0.505346
3,0.764500,3.218640,0.956633,0.476164
4,0.642500,3.390482,0.924320,0.462979
5,0.596400,3.692659,0.926871,0.468285
6,0.497000,3.609217,0.910714,0.457834
7,0.451800,3.812428,0.900085,0.460166
8,0.436100,3.875207,0.902211,0.460568


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: N


Training finished.
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/checkpoint-660
Best validation CER: 0.45783423104751186


In [ ]:
# Cell 20 — Show and save epoch-by-epoch training results

rows = []
last_train_loss = None

for log in trainer.state.log_history:
    if (
        "loss" in log
        and "eval_loss" not in log
    ):
        last_train_loss = log["loss"]

    if "eval_loss" in log:
        rows.append({
            "epoch": log.get("epoch"),
            "training_loss": last_train_loss,
            "validation_loss": log.get("eval_loss"),
            "WER": log.get("eval_wer"),
            "CER": log.get("eval_cer"),
        })

history_df = pd.DataFrame(rows)

display(history_df)

HISTORY_PATH = RESULTS_DIR / "training_history.csv"

history_df.to_csv(
    HISTORY_PATH,
    index=False,
)

print("Saved:", HISTORY_PATH)


,epoch,training_loss,validation_loss,WER,CER
0,1.0,2.9733,3.264127,0.997449,0.991961
1,2.0,1.2989,3.276336,1.000425,0.505346
2,3.0,0.7645,3.218640,0.956633,0.476164
3,4.0,0.6425,3.390482,0.924320,0.462979
4,5.0,0.5964,3.692659,0.926871,0.468285
5,6.0,0.4970,3.609217,0.910714,0.457834
6,7.0,0.4518,3.812428,0.900085,0.460166
7,8.0,0.4361,3.875207,0.902211,0.460568


Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/omniASR_w2v_300m_tarifit_v1_2/training_history.csv


In [ ]:
# Cell 21 — Evaluate the selected best checkpoint on validation

best_metrics = trainer.evaluate(
    eval_dataset=val_ds
)

best_wer = best_metrics["eval_wer"]
best_cer = best_metrics["eval_cer"]

print(
    f"Best validation WER: {best_wer:.6f} "
    f"({best_wer * 100:.2f}%)"
)
print(
    f"Best validation CER: {best_cer:.6f} "
    f"({best_cer * 100:.2f}%)"
)
print(
    "Selected checkpoint:",
    trainer.state.best_model_checkpoint,
)


Best validation WER: 0.910714 (91.07%)
Best validation CER: 0.457834 (45.78%)
Selected checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/checkpoint-660


In [ ]:
# Cell 22 — Save the selected OmniASR model and processor

BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

trainer.save_model(
    str(BEST_MODEL_DIR)
)

processor.save_pretrained(
    str(BEST_MODEL_DIR)
)

print("Saved selected model:", BEST_MODEL_DIR)


Saved selected model: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/best_model


In [ ]:
# Cell 23 — Save validation predictions for qualitative error analysis

prediction_output = trainer.predict(
    val_ds
)

pred_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

label_ids = prediction_output.label_ids.copy()
label_ids[
    label_ids == -100
] = tokenizer.pad_token_id

predictions = [
    x.strip()
    for x in processor.batch_decode(pred_ids)
]

references = [
    x.strip()
    for x in processor.batch_decode(
        label_ids,
        group_tokens=False,
    )
]

predictions_df = pd.DataFrame({
    "segment_id": val_ds["segment_id"],
    "reference": references,
    "prediction": predictions,
})

PREDICTIONS_PATH = (
    RESULTS_DIR
    / "validation_predictions.csv"
)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(predictions_df.head(20))

print("Saved:", PREDICTIONS_PATH)


,segment_id,reference,prediction
0,REC090_SEG0010,ssalamuɛlikum necc meryem,ssalam uɛlikum nec meryam
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,aqqay ruxxa ṭnin uɛecrin sana d ihulanda
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,mercex ag mmis n ejjiran usiɣedzi lmeɣri umi r...
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,umi usiɣedda ufi ufix mana yenni wadji ca mir ...
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,necc mammamcira djix di lmeɣrib wadji mana yen...
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,necdi lmeɣrib ira ɣari lḥurriya yinu ilaɣari i...
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,ḍinmaɣarim neccin mamciraniɛic
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,a babadimma waɣanexca ɣanex ca neḥwayejj n teg...
8,REC090_SEG0018,lmuhim wsiɣd,muhim usiɣt
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,neccira ɛemas wawsiɣ d ɣa uruppa wsiɣeddit iya...


Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/omniASR_w2v_300m_tarifit_v1_2/validation_predictions.csv


In [ ]:
# Cell 24 — Save the complete experiment summary

summary = {
    "experiment": "OmniASR-W2V-300M -> Tarifit V1.2 CTC fine-tuning",
    "base_model": BASE_MODEL_ID,
    "data_version": "V1.2",
    "metadata_sha256": sha256,
    "train_segments": len(train_ds),
    "validation_segments": len(val_ds),
    "train_hours": train_duration["hours"],
    "validation_hours": val_duration["hours"],
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "augmentation": "none",
    "feature_extractor_frozen": True,
    "encoder_finetuned": True,
    "seed": SEED,
    "max_epochs": 8,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "physical_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "early_stopping_patience": 2,
    "checkpoint_selection_metric": "CER",
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_wer": float(best_wer),
    "best_validation_cer": float(best_cer),
}

SUMMARY_PATH = (
    RESULTS_DIR
    / "experiment_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(summary, indent=2))
print("\nSaved:", SUMMARY_PATH)


## Interpretation

This experiment should be compared primarily with:

- the earlier OmniASR V1.1 fine-tuning experiment, to assess the effect of the refined V1.2 corpus; and
- the final MMS V1.2 systems, to compare model families on the same frozen V1.2 development split.

Do **not** evaluate repeatedly on the held-out test set while choosing hyperparameters. Final test evaluation should happen only after the test references are manually finalized and frozen.
